# Script to find the transformation between session maps 

In [9]:
from pathlib import Path
import numpy as np
from scipy.spatial.transform import Rotation as R

import os
import glob

from dataclasses import dataclass
import numpy as np
import open3d as o3d

# Pathes and Parameters

In [10]:
BASE_DIR = Path("/home/vorart/workspace/Sber/foundation_ssd/foundation_data/maps/mmpr_dataset/")
MAP_1_PATH = BASE_DIR / "map1" / "keyframe_map_downsample_90"
MAP_2_PATH = BASE_DIR / "map2" / "keyframe_map_downsample_90"
MAP_3_PATH = BASE_DIR / "map3" / "keyframe_map_downsample_90"
MAP_4_PATH = BASE_DIR / "map4" / "keyframe_map_downsample_90"
MAP_5_PATH = BASE_DIR / "map5" / "keyframe_map_downsample_90"
MAP_6_PATH = BASE_DIR / "map6" / "keyframe_map_downsample_90"
MAP_7_PATH = BASE_DIR / "map7" / "keyframe_map_downsample_90"
MAP_8_PATH = BASE_DIR / "map8" / "keyframe_map_downsample_90"
MAP_9_PATH = BASE_DIR / "map_test" / "keyframe_map_downsample_90"

# Display original session maps

In [11]:
def load_poses(poses_file):
    poses = {}
    with open(poses_file) as f:
        for idx, ln in enumerate(f):
            if ln.startswith('#') or not ln.strip():
                continue
            t, x, y, z, qx, qy, qz, qw = map(float, ln.strip().split(','))
            q = np.array([qw, qx, qy, qz])
            R = o3d.geometry.get_rotation_matrix_from_quaternion(q)
            T = np.eye(4)
            T[:3, :3] = R
            T[:3, 3] = [x, y, z]
            poses[idx] = T
    return poses

def load_point_clouds(scans_dir, poses):
    pcd_dict = {}
    for pcd_path in sorted(glob.glob(os.path.join(scans_dir, "*.pcd"))):
        try:
            idx = int(os.path.basename(pcd_path).split('.')[0])
        except ValueError:
            continue
        if idx not in poses:
            continue
        pc = o3d.io.read_point_cloud(pcd_path)
        pc.transform(poses[idx])
        pcd_dict[idx] = pc
    return pcd_dict

def load_keyframemap(path):
    map_path = Path(path)
    poses = load_poses(map_path / "poses.csv" )
    pcd_dict = load_point_clouds(map_path / "scans" , poses)
    return pcd_dict

def merge_full(pcd_dict):
    m = o3d.geometry.PointCloud()
    for scan in pcd_dict.values():
        m += scan
    return m

def load_map(path, voxelize=None):
    pcd = merge_full(load_keyframemap(path))
    return pcd.voxel_down_sample(voxelize) if voxelize else pcd

In [12]:
map1 = load_map(MAP_1_PATH, voxelize=0.4)
map2 = load_map(MAP_2_PATH, voxelize=0.4)
map3 = load_map(MAP_3_PATH, voxelize=0.4)

map1.paint_uniform_color([1,0,0])
map2.paint_uniform_color([0,1,0])
map3.paint_uniform_color([0,0,1])
o3d.visualization.draw_geometries([map1, map2, map3])

# Define the Local Registrator (ICP)

In [36]:
@dataclass
class ICPParams:
    voxel: float = 0.2                      # used for downsample + normal radius
    max_correspondence_distance: float = 2.0
    max_iterations: int = 60
    rel_fitness: float = 1e-6
    rel_rmse: float   = 1e-6
    use_point_to_plane: bool = True         # default to point-to-plane

In [14]:
class LocalRegistrator:
    """
    Minimal registrar Uses Open3D's ICP (point-to-plane by default).
    """
    class _ICPResult:
        """Tiny wrapper to support .T_target_source"""
        def __init__(self, T, fitness, rmse):
            self.T_target_source = T
            self.fitness = fitness
            self.rmse = rmse

    def __init__(self, params: ICPParams):
        self.params = params
        self.source = None   # open3d.geometry.PointCloud
        self.target = None

    @staticmethod
    def _to_pcd(points: np.ndarray) -> o3d.geometry.PointCloud:
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(points[:, :3])
        return pcd

    def _prep(self, pcd: o3d.geometry.PointCloud) -> o3d.geometry.PointCloud:
        # Downsample + normals once for stable point-to-plane ICP
        v = self.params.voxel
        p = pcd.voxel_down_sample(v) if v and v > 0 else pcd
        o3d.geometry.PointCloud.estimate_normals(
            p,
            search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=2.5*v, max_nn=50)
        )
        p.orient_normals_consistent_tangent_plane(30)
        return p

    def set_target(self, target_points: np.ndarray) -> None:
        self.target = self._prep(self._to_pcd(target_points))

    def set_source(self, source_points: np.ndarray) -> None:
        self.source = self._prep(self._to_pcd(source_points))

    def _icp(self, init_T: np.ndarray) -> _ICPResult:
        if self.source is None or self.target is None:
            raise ValueError("Source/Target not set")

        # Loss + criteria
        loss = o3d.pipelines.registration.HuberLoss(k=1.0)
        criteria = o3d.pipelines.registration.ICPConvergenceCriteria(
            relative_fitness=self.params.rel_fitness,
            relative_rmse=self.params.rel_rmse,
            max_iteration=self.params.max_iterations,
        )
        if self.params.use_point_to_plane:
            estimation = o3d.pipelines.registration.TransformationEstimationPointToPlane(loss)
        else:
            estimation = o3d.pipelines.registration.TransformationEstimationPointToPoint(loss)

        result = o3d.pipelines.registration.registration_icp(
            source=self.source,
            target=self.target,
            max_correspondence_distance=self.params.max_correspondence_distance,
            init=init_T,
            estimation_method=estimation,
            criteria=criteria,
        )
        return self._ICPResult(result.transformation, result.fitness, result.inlier_rmse)

    def align(self, init_transformation: np.ndarray | None = None) -> _ICPResult:
        if init_transformation is None:
            init_transformation = np.eye(4)
        return self._icp(init_transformation)

    def align_one(self, source: np.ndarray, target: np.ndarray, init_transformation=None) -> _ICPResult:
        self.set_source(source); self.set_target(target)
        return self.align(init_transformation)

    def align_clouds(self, source_o3d: o3d.geometry.PointCloud,
                     target_o3d: o3d.geometry.PointCloud,
                     init_transformation=None) -> _ICPResult:
        s = np.asarray(source_o3d.points)[:, :3]
        t = np.asarray(target_o3d.points)[:, :3]
        return self.align_one(s, t, init_transformation)

## Function to display the registration result

In [15]:
import copy, numpy as np, open3d as o3d
try: import glfw; K_RIGHT, K_LEFT = glfw.KEY_RIGHT, glfw.KEY_LEFT
except ModuleNotFoundError: K_RIGHT, K_LEFT = 262, 263          # fallback

_GRN, _GRY = [0, 1, 0], [.6, .6, .6]

def _pc(pts, col):
    g = o3d.geometry.PointCloud()
    g.points = o3d.utility.Vector3dVector(pts)
    g.paint_uniform_color(col)
    return g

def show_registration(src_points, tgt_points, T_list, label="Registration viewer"):
    if not isinstance(T_list, list):
        T_list = [T_list]

    src = _pc(src_points, _GRN)
    tgt = _pc(tgt_points, _GRY)

    vis = o3d.visualization.VisualizerWithKeyCallback()
    vis.create_window(label, 1280, 720)
    opt = vis.get_render_option()
    opt.point_size, opt.background_color = 4.0, np.zeros(3)

    state = {"i": len(T_list)-1, "α": 1.0}

    def _blend_seq(i, α):
        src_local = copy.deepcopy(src)
        Ti = np.eye(4)
        for j in range(i):
            Ti = T_list[j] @ Ti

        T_interp = np.eye(4)
        T_interp[:3, :3] = (1 - α) * np.eye(3) + α * T_list[i][:3, :3]
        T_interp[:3, 3] = α * T_list[i][:3, 3]

        src_local.transform(Ti)
        src_local.transform(T_interp)
        src_local.paint_uniform_color([1 - α, α, 0])
        return src_local
    
    step_geom = _blend_seq(state["i"], state["α"])

    vis.add_geometry(tgt)
    vis.add_geometry(step_geom)  # initial full transform to allocate buffer

    def _refresh():
        cam = vis.get_view_control().convert_to_pinhole_camera_parameters()
        vis.clear_geometries(); vis.add_geometry(tgt); vis.add_geometry(step_geom)
        vis.get_view_control().convert_from_pinhole_camera_parameters(cam)
        vis.poll_events(); vis.update_renderer()

    def inc_alpha(vis, action, mods):
        if state["i"] == len(T_list) - 1 and state["α"] >=+ 1.0:
            return False
        state["α"] += 0.05
        if state["α"] >=+ 1.0:
            state["α"] = 0.0 if state["i"] < len(T_list) - 1 else 1.0
            state["i"] += 1 if state["i"] < len(T_list) - 1 else 0
        blended = _blend_seq(state["i"], state["α"])
        step_geom.points, step_geom.colors = blended.points, blended.colors
        _refresh(); return False

    def dec_alpha(vis, action, mods):
        if state["i"] == 0 and state["α"] <= 0.0:
            return False
        state["α"] -= 0.05
        if state["α"] <= 0.0:
            state["α"] = 1.0 if state["i"] > 0 else 0.0
            state["i"] -= 1 if state["i"] > 0 else 0
        blended = _blend_seq(state["i"], state["α"])
        step_geom.points, step_geom.colors = blended.points, blended.colors
        _refresh(); return False

    vis.register_key_action_callback(K_RIGHT, inc_alpha)
    vis.register_key_action_callback(K_LEFT,  dec_alpha)

    # print("\n← / → : blend transformation\n")
    _refresh(); vis.run(); vis.destroy_window()


# Define the overlapping between the maps

Specify overlapping regions between maps for registration
Each dictionary describes:
  - `src_map`: index of source map
  - `tgt_map`: index of target map
  - `src_range`: range of frames in source map
  - `tgt_range`: range of frames in target map
  - `init_rotation_yaw`: initial guess for yaw rotation (in radians)

>! Note: we assume that overlap links goes in accending order from 0 to N without skiping

1-2: (1545, 1713):(453,550)
1-3: (1040, 1213):(150,350)
1-4: (30, 366):(160,350)


In [17]:
map1 = load_map(MAP_1_PATH, voxelize=0.3)
map2 = load_map(MAP_2_PATH, voxelize=0.3)

map1.paint_uniform_color([1,0,0])
map2.paint_uniform_color([0,1,0])
o3d.visualization.draw_geometries([map1, map2])

In [48]:
import matplotlib.pyplot as plt

class MapAligner:
    def __init__(self, map_paths, map_overlaps, registrator, voxel=1.0):
        """
        :param map_paths: list of Paths to maps
        :param map_overlaps: list of dicts:
            {
                "src_map": int,
                "tgt_map": int,
                "src_range": (start, end),
                "tgt_range": (start, end),
                "init_rotation_yaw": float # default = 0 radians
                "center_align": bool # default = True
            }
        :param voxel: voxel size for registration
        
        # Note: we assume that overlap links goes in accending order from 0 to N without skiping
        """
        self.map_paths = map_paths
        self.map_overlaps = map_overlaps
        self.voxel = voxel
        self.registator = registrator
        self.pcd_maps = [load_keyframemap(p) for p in map_paths]
        self.transforms = [np.eye(4) for _ in map_paths]  # T_map_i_to_map0

    def merge(self, ids, pcd_dict):
        m = o3d.geometry.PointCloud()
        for i in ids:
            m += pcd_dict[i]
        return m

    def get_transformation(self):
        return self.transforms

    def align_all(self):
        for overlap in self.map_overlaps:
            s_id, t_id = overlap["src_map"], overlap["tgt_map"]
            s_rng, t_rng = overlap["src_range"], overlap["tgt_range"]
            init_yaw = overlap.get("init_rotation_yaw", 0.0)  # default = 0 radians
            center_align = overlap.get("center_align", True)  # default = True
            
            print(f"\n# == Aligning 'map {s_id}' to 'map {t_id}' ==")
            
            src = self.merge(range(*s_rng), self.pcd_maps[s_id]).voxel_down_sample(self.voxel)
            tgt = self.merge(range(*t_rng), self.pcd_maps[t_id]).voxel_down_sample(self.voxel)

            # Create initial rotation around Z axis (yaw)
            src_rotated = o3d.geometry.PointCloud(src)
            T_rotate = np.eye(4)

            if abs(init_yaw) > 0.:
                c, s = np.cos(init_yaw), np.sin(init_yaw)
                T_rotate[:3, :3] = [
                    [c, -s, 0],
                    [s,  c, 0],
                    [0,  0, 1],
                ]
                src_rotated.transform(T_rotate)

            # Translate to the center of mass
            src_transformed = o3d.geometry.PointCloud(src_rotated)
            T_shift = np.eye(4)
            if center_align:
                T_shift[:3, 3] = tgt.get_center() - src_rotated.get_center()
                src_transformed.transform(T_shift)

            T = self.registator.align_clouds(src_transformed, tgt).T_target_source
            
            show_registration(np.asarray(src.points), np.asarray(tgt.points), [T_rotate, T_shift, T])
            T_final = T @ T_shift @ T_rotate

            with np.printoptions(precision=12, suppress=False, linewidth=160):
                print(f"T_final_{s_id}_{t_id} = [")
                for row in T_final:
                    print("[", end="")
                    print(", ".join(f"{v: .12f}" for v in row), end="")
                    print("],")
                print("]")
                    
            # Final transform: src_map → tgt_map
            self.transforms[t_id] = self.transforms[s_id] @ np.linalg.inv(T_final)

        print(f"\n# == Computed transformation to the map0' ==")
        for i, T in enumerate(self.get_transformation()):
            if i == 0: continue
            with np.printoptions(precision=12, suppress=False, linewidth=160):
                print(f"T_{i}_0 = [")
                for row in T:
                    print("[", end="")
                    print(", ".join(f"{v: .12f}" for v in row), end="")
                    print("],")
                print("]")

    def get_merged(self, downsample=True):
        merged = []
        colors = self._generate_colors(len(self.pcd_maps))

        for i, (pcd_dict, color) in enumerate(zip(self.pcd_maps, colors)):
            full_map = o3d.geometry.PointCloud()
            for pc in pcd_dict.values():
                full_map += pc
            if downsample:
                full_map = full_map.voxel_down_sample(self.voxel)
            full_map.paint_uniform_color(color)
            full_map.transform(self.transforms[i])
            merged.append(full_map)
        return merged
    
    def show_merged(self, downsample=True):
        o3d.visualization.draw_geometries(self.get_merged(downsample))

    def _generate_colors(self, n):
        cmap = plt.get_cmap('tab10')
        return [list(cmap(i % 10)[:3]) for i in range(n)]


## Align maps
> Use the arrows "<-" and "->" to analyze the success of registration

In [49]:
map_paths = [MAP_1_PATH, MAP_2_PATH, MAP_3_PATH, MAP_4_PATH]
map_overlaps = [
    {"src_map": 0, "tgt_map": 1, "src_range": (1545, 1713),  "tgt_range": (453,550), "init_rotation_yaw": 0},  # map1 vs map2
    {"src_map": 0, "tgt_map": 2, "src_range": (1040, 1213), "tgt_range": (150,350),  "init_rotation_yaw": 0, "center_align": False},  # map1 vs map3
    {"src_map": 0, "tgt_map": 3, "src_range": (30, 366), "tgt_range": (160,350),     "init_rotation_yaw": 0},  # map1 vs map4
]

In [50]:
local_registrator = LocalRegistrator(ICPParams())
aligner = MapAligner(map_paths, map_overlaps, local_registrator, voxel=0.3)
aligner.align_all()


# == Aligning 'map 0' to 'map 1' ==
T_final_0_1 = [
[ 0.957110281871, -0.289398175442,  0.013733331252,  0.235764913149],
[ 0.289258606145,  0.957184419292,  0.011289208796, -0.571967798002],
[-0.016412407127, -0.006832533557,  0.999841962201,  0.068496062721],
[ 0.000000000000,  0.000000000000,  0.000000000000,  1.000000000000],
]

# == Aligning 'map 0' to 'map 2' ==
T_final_0_2 = [
[ 0.994854030000,  0.101285918635, -0.002573262206, -0.581124270317],
[-0.101284597699,  0.994857287291,  0.000638899354, -0.424919328239],
[ 0.002624740166, -0.000374979770,  0.999996485058,  0.121358373403],
[ 0.000000000000,  0.000000000000,  0.000000000000,  1.000000000000],
]

# == Aligning 'map 0' to 'map 3' ==
T_final_0_3 = [
[ 0.871448913817,  0.489922660676,  0.023506959864, -2.826938788496],
[-0.490183542743,  0.871593865811,  0.006650376246, -1.712043124308],
[-0.017230351997, -0.017318188021,  0.999701553132,  0.125152681817],
[ 0.000000000000,  0.000000000000,  0.000000000000,  1.000000000000

In [ ]:
# == Computed transformation to the map1' ==
T_2_1 = [
[ 0.956711209165,  0.290813077207, -0.011463698546, -0.153455820707],
[-0.290812256493,  0.956778490454,  0.001775296591,  0.688947242214],
[ 0.011484499655,  0.001635337894,  0.999932713705, -0.077340220372],
[ 0.000000000000,  0.000000000000,  0.000000000000,  1.000000000000],
]
T_3_1 = [
[ 0.994962719425, -0.100235832042, -0.001401759667,  0.514291026771],
[ 0.100237186633,  0.994963134041,  0.000931834714,  0.461334808929],
[ 0.001301295964, -0.001067649246,  0.999998583376, -0.130197717405],
[ 0.000000000000,  0.000000000000,  0.000000000000,  1.000000000000],
]
T_4_1 = [
[ 0.802577217171, -0.595935887259, -0.027022745116,  0.386908564150],
[ 0.595360363748,  0.803014107814, -0.026727886735,  0.886000197368],
[ 0.037627752456,  0.005362921595,  0.999277434608, -0.097609346688],
[ 0.000000000000,  0.000000000000,  0.000000000000,  1.000000000000],
]


1-5: (366,703):(1,535) np.pi/2
1-6: (600,871):(1,450) np.pi/2
1-7: (2000,2555):(1,1180) np.pi/2
1-8: (2900,3229):(1,906) np.pi

In [47]:
map_paths = [MAP_1_PATH, MAP_5_PATH, MAP_6_PATH, MAP_7_PATH, MAP_8_PATH]
map_overlaps = [
    {"src_map": 0, "tgt_map": 1, "src_range": (366,703),  "tgt_range": (230,535), "init_rotation_yaw":  -2.3*np.pi/4, "center_align": True},  # map1 vs map2
    {"src_map": 0, "tgt_map": 2, "src_range": (600,871), "tgt_range": (1,450),  "init_rotation_yaw": -3*np.pi/4, "center_align": True},  # map1 vs map3
    {"src_map": 0, "tgt_map": 3, "src_range": (2000,2555), "tgt_range": (1,1180),     "init_rotation_yaw": -3*np.pi/4, "center_align": True},  # map1 vs map4
    {"src_map": 0, "tgt_map": 4, "src_range": (2900,3229), "tgt_range": (1,906),     "init_rotation_yaw": np.pi, "center_align": True},  # map1 vs map4
]
aligner = MapAligner(map_paths, map_overlaps, local_registrator, voxel=0.3)
aligner.align_all()


# == Aligning 'map 0' to 'map 1' ==
T_final_0_1 = [
[-0.314015910006,  0.949408164245,  0.004259803837, -4.769583631077],
[-0.949409051338, -0.313990708099, -0.005682294081,  21.267493087907],
[-0.004057277569, -0.005828627066,  0.999974782485,  0.603670687303],
[ 0.000000000000,  0.000000000000,  0.000000000000,  1.000000000000],
]

# == Aligning 'map 0' to 'map 2' ==
T_final_0_2 = [
[-0.666876352700,  0.745168270604,  0.000422723393,  4.191889597814],
[-0.745157342238, -0.666869447050,  0.005067139233,  21.600445717283],
[ 0.004057772695,  0.003064159890,  0.999987072619,  0.316055970128],
[ 0.000000000000,  0.000000000000,  0.000000000000,  1.000000000000],
]

# == Aligning 'map 0' to 'map 3' ==
T_final_0_3 = [
[-0.608371581503,  0.793644427264, -0.003541453394,  2.441310154665],
[-0.793649920732, -0.608374670851,  0.000251373208,  21.921133396174],
[-0.001955029598,  0.002963602522,  0.999993697440,  0.464536691305],
[ 0.000000000000,  0.000000000000,  0.000000000000,  1.000000000

In [53]:
# == Computed transformation to the map0' ==
T_5_1 = [
[-0.314015910006, -0.949409051338, -0.004057277569,  18.696274552193],
[ 0.949408164245, -0.313990708099, -0.005828627066,  11.209595424855],
[ 0.004259803837, -0.005682294081,  0.999974782485, -0.462489823486],
[ 0.000000000000,  0.000000000000,  0.000000000000,  1.000000000000],
]
T_6_1 = [
[-0.666876352700, -0.745157342238,  0.004057772695,  18.889920284467],
[ 0.745168270604, -0.666869447050,  0.003064159890,  11.280045723329],
[ 0.000422723393,  0.005067139233,  0.999987072619, -0.427276360094],
[ 0.000000000000,  0.000000000000,  0.000000000000,  1.000000000000],
]
T_7_1 = [
[-0.608371581503, -0.793649920732, -0.001955029598,  18.883837684948],
[ 0.793644427264, -0.608374670851,  0.002963602522,  11.397353413000],
[-0.003541453394,  0.000251373208,  0.999993697440, -0.461398363017],
[ 0.000000000000,  0.000000000000,  0.000000000000,  1.000000000000],
]
T_8_1 = [
[-0.989549287520, -0.144184982476,  0.001702467792,  16.858101263295],
[ 0.144180168093, -0.989547836670, -0.002675456918, -11.745636334855],
[ 0.002070434030, -0.002402034395,  0.999994971754, -0.467016966506],
[ 0.000000000000,  0.000000000000,  0.000000000000,  1.000000000000],
]

______________________
Check
______________________


In [54]:
aligner.show_merged()

In [ ]:
maps = [None]*8

# maps[0] = load_map(MAP_1_PATH, voxelize=0.3).transform(np.eye(4))
maps[1] = load_map(MAP_2_PATH, voxelize=0.3).transform(T_2_1)
# maps[2] = load_map(MAP_3_PATH, voxelize=0.3).transform(T_3_1)
# maps[3] = load_map(MAP_4_PATH, voxelize=0.3).transform(T_4_1)
# maps[4] = load_map(MAP_5_PATH, voxelize=0.3).transform(T_5_1)
# maps[5] = load_map(MAP_6_PATH, voxelize=0.3).transform(T_6_1)
# maps[6] = load_map(MAP_7_PATH, voxelize=0.3).transform(T_7_1)
maps[7] = load_map(MAP_8_PATH, voxelize=0.3).transform(T_8_1)

cmap = plt.get_cmap('tab10')
colors = [list(cmap(i % 10)[:3]) for i in range(len(maps))]
colored_maps = []
for i, m in enumerate(maps):
    if m:
        m.paint_uniform_color(colors[i])
        colored_maps.append(m)

o3d.visualization.draw_geometries(colored_maps)

: 